In [ ]:
import numpy as np
from tqdm import tqdm

import librosa
import os
import numpy as np
from pydub import AudioSegment
import soundfile as sf
from pydub.silence import detect_silence

### pre-processing

In [ ]:
import numpy as np
import pandas as pd


all = pd.read_csv('......../Downloads/20240128_metadata15075.csv')
all

In [ ]:
audio_dataset_path = '......../Downloads/Tacticas Audio zip/version11012024/Tacticas Audio/Tacticas Audio Compressed/'
output_folder = '......../Downloads/Tacticas Audio zip/version11012024/Tacticas Audio/all/'

for index_num,row in tqdm(all.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),'uid0000' + str(row['sid']) + '/' + row['new_filename'])
    output_path = os.path.join(os.path.abspath(output_folder),row['new_filename'])
    if os.path.exists(file_name):
        y, sr = librosa.load(file_name)
        length = librosa.get_duration(y=y, sr=sr)
        if length > 0.2:
            sf.write(output_path, y, sr)


In [ ]:
audio_dataset_path = '......../Downloads/Tacticas Audio zip/version11012024/Tacticas Audio/all/'
output_folder = '......../Downloads/Tacticas Audio zip/version11012024/Tacticas Audio/20240123_remove_slience'

for index_num,row in tqdm(all.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),row['new_filename'])
    output_path = os.path.join(os.path.abspath(output_folder),row['new_filename'])
    if os.path.exists(file_name):
        audio = AudioSegment.from_wav(file_name)
        silence_segments = detect_silence(audio, min_silence_len= 1000, silence_thresh=-50)
        silence_segments_ms = [(start, stop) for start, stop in silence_segments]
        non_silent_segments = []
        if silence_segments_ms == []:
            audio.export(output_path, format="wav")
        elif silence_segments_ms != []:
            for i, (start, stop) in enumerate(silence_segments_ms):
                if i == 0:
                    non_silent_segments.append(audio[:start])
                if i < len(silence_segments_ms) - 1:
                    non_silent_segments.append(audio[stop:silence_segments_ms[i + 1][0]])
            non_silent_segments.append(audio[silence_segments_ms[-1][1]:])
            non_silent_audio = sum(non_silent_segments)
            non_silent_audio.export(output_path, format="wav")